# Task 2.2 — Core Reproduction

**Contribution reproduced:** Fast Shapelet Discovery (Algorithm 6) from the paper, using **Efficient Distance Computation** (Algorithm 4, sufficient statistics) and **Candidate Pruning** (Algorithm 5). We also use information gain (Definition 3) and z-normalized subsequence distance (Eq. 2).

**Evaluation metric:** Accuracy on the test set; information gain of the chosen shapelet.

## Load data and hyperparameters

In [1]:
RANDOM_STATE = 42
import numpy as np
import sys
import os
# Ensure partB is on path so 'import shapelets' finds partB/shapelets.py (e.g. when kernel cwd is parent)
sys.path.insert(0, os.getcwd())
_partb = os.path.join(os.getcwd(), 'partB')
if os.path.isdir(_partb) and not os.path.isfile(os.path.join(os.getcwd(), 'shapelets.py')):
    sys.path.insert(0, _partb)

# Load the toy dataset (from Task 2.1); fallback to generating if not found
def load_toy_dataset(data_path='data/toy_ts.npz'):
    if os.path.exists(data_path):
        data = np.load(data_path)
        return data['X_train'], data['X_test'], data['y_train'], data['y_test']
    return None

loaded = load_toy_dataset()
if loaded is not None:
    X_train, X_test, y_train, y_test = loaded
else:
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    X, y = make_classification(n_samples=120, n_features=50, n_informative=10, n_redundant=5, n_classes=2, random_state=RANDOM_STATE)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# Hyperparameters (single place — Reproducibility Checklist)
MIN_LEN = 3
MAX_LEN = 8
STEP = 2
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (96, 50) Test: (24, 50)


Data is loaded as in Task 2.1. Hyperparameters control the shapelet search: minimum and maximum subsequence length and step between lengths (paper Algorithm 6, loops over length \(l\)).

## Helper: Efficient Distance Computation (Algorithm 4)

The paper uses **sufficient statistics** (Section 3.1) to compute window mean and variance in O(1) per window instead of recomputing from scratch. We precompute cumulative sums \(S_x\) and \(S_{x^2}\) so that for any window \([i, i+L)\), \(\sum x = S_x[i+L] - S_x[i]\) and \(\sum x^2 = S_{xx}[i+L] - S_{xx}[i]\).

In [2]:
import numpy as np
# Define helpers here (Algorithm 4) so we don't depend on importing them from shapelets
def z_norm(x):
    x = np.asarray(x, dtype=float)
    if x.std() < 1e-10:
        return np.zeros_like(x)
    return (x - x.mean()) / x.std()

def compute_sufficient_statistics(series):
    '''Cumulative sums S_x, S_xx for window mean/variance. Paper Section 3.1.'''
    series = np.asarray(series, dtype=float)
    S_x = np.zeros(len(series) + 1)
    for i in range(1, len(series) + 1):
        S_x[i] = S_x[i - 1] + series[i - 1]
    S_xx = np.zeros(len(series) + 1)
    for i in range(1, len(series) + 1):
        S_xx[i] = S_xx[i - 1] + series[i - 1] ** 2
    return S_x, S_xx

def sdist_efficient(shapelet, series, S_x, S_xx):
    '''Min z-normalized distance using sufficient stats. Paper Algorithm 4.'''
    shapelet = np.asarray(shapelet, dtype=float)
    L = len(shapelet)
    n = len(series)
    if n < L:
        return float('inf')
    s_norm = z_norm(shapelet)
    dot_products = np.correlate(series, s_norm, mode='valid')
    min_dist = np.inf
    for start in range(n - L + 1):
        window_sum = S_x[start + L] - S_x[start]
        window_sum_sq = S_xx[start + L] - S_xx[start]
        window_mean = window_sum / L
        window_std = np.sqrt(max(0.0, (window_sum_sq / L) - (window_mean ** 2)))
        if window_std < 1e-10:
            dist = np.inf
        else:
            correlation = dot_products[start] / (L * window_std)
            dist = np.sqrt(max(0.0, 2.0 * (1.0 - correlation)))
        if dist < min_dist:
            min_dist = dist
    return min_dist

import sys, os
sys.path.insert(0, os.getcwd())
_partb = os.path.join(os.getcwd(), 'partB')
if os.path.isdir(_partb) and not os.path.isfile(os.path.join(os.getcwd(), 'shapelets.py')):
    sys.path.insert(0, _partb)
from shapelets import sdist

# Example: compute S_x, S_xx for one series (paper Section 3.1)
example_series = X_train[0]
S_x, S_xx = compute_sufficient_statistics(example_series)
print('S_x length:', len(S_x), '(one more than series length)')
print('First few S_x:', S_x[:5])

candidate = X_train[0, 5:10]
d_efficient = sdist_efficient(candidate, example_series, S_x, S_xx)
d_naive = sdist(candidate, example_series)
print('Distance (efficient):', d_efficient, ' (naive):', d_naive)

S_x length: 51 (one more than series length)
First few S_x: [ 0.         -0.10629091  0.49842614  1.86621055  3.40004319]
Distance (efficient): 0.0  (naive): 0.0


This block computes the cumulative sums \(S_x\) and \(S_{xx}\) as defined in Section 3.1 of the paper. The efficient distance function uses these to compute the z-normalized distance from the shapelet to every window without recomputing mean/variance in a loop.

## Helper: Candidate Pruning (Algorithm 5)

We skip candidates that **cannot** beat the current best information gain (admissible pruning). A simple case: if the distance vector has almost no variation, no threshold can achieve a good split, so we prune.

In [3]:
from shapelets import can_prune_candidate, best_ig_threshold

# Example: a candidate that gives almost identical distances to all series will be pruned
distances_low_var = np.ones(100) * 0.5  # no variation
labels = np.random.randint(0, 2, 100)
prune = can_prune_candidate(distances_low_var, labels, current_best_ig=0.1)
print('Prune low-variance distances?', prune)

# A candidate with spread distances might not be pruned
distances_high_var = np.random.rand(100) * 2
prune2 = can_prune_candidate(distances_high_var, labels, current_best_ig=0.1)
print('Prune high-variance distances?', prune2)

Prune low-variance distances? True
Prune high-variance distances? False


Algorithm 5 in the paper uses admissible pruning so we never discard a candidate that could be optimal; we only skip when the distance vector has zero (or negligible) variance.

## Fast Shapelet Discovery (Algorithm 6)

In [4]:
from shapelets import discover_shapelet_fast, get_split_labels, predict_shapelet

# Run Fast Shapelet Discovery with both efficient distance and pruning enabled
best_shapelet, best_tau, best_ig, best_gap, best_distances = discover_shapelet_fast(
    X_train, y_train,
    min_len=MIN_LEN, max_len=MAX_LEN, step=STEP,
    random_state=RANDOM_STATE,
    use_efficient_dist=True,
    use_pruning=True
)
print('Best information gain:', best_ig)
print('Best threshold tau:', best_tau)
print('Shapelet length:', len(best_shapelet))

Best information gain: 1.0
Best threshold tau: 0.34308157786234483
Shapelet length: 7


Algorithm 6 iterates over all candidate subsequences (from each series and each length). For each candidate it computes distances using Algorithm 4 (sufficient statistics), then applies Algorithm 5 to prune if the candidate cannot beat the current best. The best threshold and information gain are computed as in Algorithm 3 (orderline, midpoints, Definition 3).

## Class mapping and prediction (distance threshold split)

In [5]:
# Determine which class corresponds to "close" (distance <= tau) and "far" by majority vote
label_close, label_far = get_split_labels(best_distances, y_train, best_tau)
y_pred = predict_shapelet(X_test, best_shapelet, best_tau, label_close, label_far)
acc = (y_pred == y_test).mean()
print('Test accuracy (single shapelet, distance threshold split):', acc)

Test accuracy (single shapelet, distance threshold split): 1.0


We determine which class corresponds to "close" (distance \(\leq \tau\)) and "far" by majority vote on the training orderline (paper: left partition vs right partition). Then we classify test points by comparing their subsequence distance to \(\tau\). Reference: Section 2 split definition.

## Result summary

In [6]:
print('Single shapelet — IG:', best_ig, 'Accuracy:', acc)

Single shapelet — IG: 1.0 Accuracy: 1.0
